<a href="https://colab.research.google.com/github/hansjio/tsektxt-model-training/blob/master/notebooks/colab_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TsekTxt: test the 3 models (Colab, GPU optional)
**What this is.** The working notebook for testing the three fine-tuned models, scoring the Taglish test set and running Integrated Gradients. It never retrains or uploads anything.
**Before you start.** Runtime > Change runtime type > T4 GPU (faster, not required). Run the cells top to bottom; sections marked optional can be skipped.
**Just presenting?** Use `colab_present.ipynb` instead (read-only, fast). Open this notebook from https://colab.research.google.com/github/hansjio/tsektxt-model-training/blob/master/notebooks/colab_eval.ipynb

In [ ]:
import os
# Safe to re-run: clone only if the folder is not there yet, and do not nest folders.
if os.path.basename(os.getcwd()) != 'tsektxt-model-training':
    if not os.path.isdir('tsektxt-model-training'):
        !git clone -q https://github.com/hansjio/tsektxt-model-training.git
    %cd tsektxt-model-training
!pip -q install 'transformers>=4.41,<5' scikit-learn pandas pyyaml scipy captum joblib

## 1. Try any text (all 3 models, loaded from Hugging Face)
**What it does:** scores one post with all three models and prints the label and P(credible) for each.
**Input:** the text inside the quotes (change it). **Output:** three lines of verdicts.
**Time:** about a minute the first time (models download), seconds after. **When:** any time, for a quick check.

In [ ]:
!python src/predict.py --text 'Totoo ba na may dayaan sa eleksyon? Share this para malaman ng lahat!'

## 2. Overfitting check: train/val/test accuracy
**What it does:** rebuilds the cleaned data and the 70/15/15 splits from `data/raw` (needed by later sections, e.g. the TF-IDF baseline). The optional last line compares accuracy on train, validation and test per model.
**Input:** `data/raw/combined_dataset_clean.csv` in the repo. **Output:** `data/processed/` splits (not committed).
**Time:** a few minutes on CPU (the optional split-gap line is slow). **When:** before Section 3b; skip if you only need Section 1.

In [ ]:
!pip -q install ftfy datasets
!python src/clean_data.py --input data/raw/combined_dataset_clean.csv
!python src/make_splits.py
# Optional, slow (runs all 3 models on every split): remove the # to run
# !python src/predict.py --split-gap

## 2a. Calibrate the confidence values (temperature scaling)
**What it does:** fits one number T per model on the internal **validation** split so the percentages better match how often each model is right. Labels never change; only percentages do. Writes `results/calibration.json` and `results/calibration_report.json` (about 5 minutes). Skipped if `results/calibration.json` already exists. After this, `predict.py`, `evaluate_external.py` (column `prob_credible_calibrated`) and `confidence_check.py` show calibrated values next to the raw ones. Never fitted on the Taglish set.

In [ ]:
import os
if not os.path.exists('results/calibration.json'):
    !python src/calibrate.py
else:
    print(open('results/calibration.json').read())

## 2b. Domain-shift checks (inference only, optional)
**What it does:** measures how accuracy changes when internal test articles are cut short, and whether a link in the text matters. Never uses the Taglish set for training or tuning.
**Input:** the splits from Section 2. **Output:** `results/domain_shift.json` / `.md` (feeds figure e).
**Time:** a few minutes, faster on GPU. **When:** once, for the paper's domain-shift figure.

In [ ]:
!python src/domain_shift_checks.py

## 3. Evaluate on the Taglish test set
**What it does:** uploads the labelled Taglish file, then scores it with each model. The Taglish set is test only: never train on it.
**Input:** `taglish_test.csv` with columns `id,text,label` (1 = credible, 0 = not credible); the upload cell asks for the file. **Output:** `results/<model>/external_*` metrics and predictions.
**Time:** a minute or two per model. **When:** only after the annotation is finished and the file exists.

In [ ]:
from google.colab import files
import os
up = files.upload()
os.makedirs('data/external', exist_ok=True)
os.replace(list(up)[0], 'data/external/taglish_test.csv')

In [ ]:
for m in ['roberta-tagalog','mbert','xlmr']:
    !python src/evaluate_external.py --model {m} --csv data/external/taglish_test.csv

## 3b. Statistics, comparison and Integrated Gradients on the Taglish set
**What it does:** runs the TF-IDF baseline on the same file, then 95% bootstrap CIs, McNemar with Holm correction, `results/comparison.md`, and Integrated Gradients (IG shows which words each model relied on).
**Input:** the Taglish file and the splits from Section 2 (the baseline trains on them). **Output:** `external_*` stats, `comparison.md`, `results/ig_*.json`.
**Time:** the IG cell is the slow one (T4 GPU recommended; CPU can take much longer). **When:** after Section 3.

In [ ]:
!python src/baseline.py --external data/external/taglish_test.csv
!python src/external_stats.py
!python src/compare.py

In [ ]:
!python src/interpret.py corpus --csv data/external/taglish_test.csv

## 3c. Results by post length (fixed bins)
**What it does:** splits the Taglish results into under 20 / 20-39 / 40-99 / 100+ words and checks how sure the models are when wrong. The primary result stays the full set; 100+ is a labelled subset.
**Input:** the Taglish file and predictions from Section 3. **Output:** `results/external_length_bins.json` / `.md` and `results/external_confidence_check.json`.
**Time:** seconds. **When:** after Section 3.

In [ ]:
!python src/length_bins.py --csv data/external/taglish_test.csv
!python src/confidence_check.py --file external_predictions.csv

## 3d. Figures for the paper
**What it does:** builds every figure whose inputs exist (PNG 300 dpi + PDF); figures with missing inputs are skipped and printed. Also writes the counts-only `results/length_distribution.json` when the data files are present.
**Input:** whatever is in `results/` so far. **Output:** `results/figures/`.
**Time:** under a minute. **When:** after Sections 3 to 3c, so the Taglish figures are filled in; it is safe to run earlier.

In [ ]:
!pip -q install matplotlib
!python src/make_figures.py

## 4. Download results
**What it does:** zips the new result files and figures so you can copy them back into your repo's `results/` folder and commit them.
**Input:** everything produced above (`results/external_*`, `results/ig_*.json`, `domain_shift.*`, `length_distribution.json`, `results/figures/`, `comparison.md`). **Output:** `external_results.zip` downloaded by the browser.
**Time:** seconds. **When:** last, once per finished run. Colab files disappear when the runtime ends, so download before closing.

In [ ]:
!zip -qr external_results.zip results/external_* results/*/external_* results/ig_*.json results/comparison.md results/domain_shift.* results/length_distribution.json results/figures
files.download('external_results.zip')